# Pedestrian Detection using PennFudanPed Dataset

Lab Assignment from [AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners).

In this lab, we will train an object detection model to detect pedestrians in images using the PennFudanPed Dataset.

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Download PennFudanPed Dataset

PennFudanPed is a pedestrian detection dataset with 170 images containing 345 labeled pedestrians. It's small and fast to download (~50MB).

In [ ]:
# Download PennFudanPed dataset manually
import urllib.request
import zipfile
import os
from tqdm import tqdm

data_dir = './data/PennFudanPed'
if not os.path.exists(data_dir):
    print("Downloading PennFudanPed dataset (~50MB)...")
    os.makedirs('./data', exist_ok=True)
    url = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
    
    # Download with progress bar
    class DownloadProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)
    
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc="PennFudanPed.zip") as pbar:
        urllib.request.urlretrieve(url, "./data/PennFudanPed.zip", reporthook=pbar.update_to)
    
    print("\nExtracting...")
    with zipfile.ZipFile("./data/PennFudanPed.zip", 'r') as zip_ref:
        zip_ref.extractall("./data")
    print("Done!")
else:
    print("Dataset already exists.")

# List dataset structure
print("\nDataset contents:")
!ls -la ./data/PennFudanPed/

In [ ]:
# Check dataset images
!ls ./data/PennFudanPed/PNGImages/ | head -5
print(f"\nTotal images: {len(os.listdir('./data/PennFudanPed/PNGImages/'))}")

## 2. Create Custom Dataset Class

In [ ]:
class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        
        # Load all image and mask files
        self.imgs = list(sorted(os.listdir(os.path.join(root, "PNGImages"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "PedMasks"))))
    
    def __getitem__(self, idx):
        # Load image and mask
        img_path = os.path.join(self.root, "PNGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "PedMasks", self.masks[idx])
        
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        
        mask = np.array(mask)
        obj_ids = np.unique(mask)
        # First id is background, remove it
        obj_ids = obj_ids[1:]
        
        # Split the color-encoded mask into a set of binary masks
        num_objs = len(obj_ids)
        masks = np.zeros((num_objs, mask.shape[0], mask.shape[1]), dtype=np.uint8)
        for i, obj_id in enumerate(obj_ids):
            masks[i] = mask == obj_id
        
        # Get bounding box coordinates from masks
        boxes = []
        for i in range(num_objs):
            pos = np.where(masks[i])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            boxes.append([xmin, ymin, xmax, ymax])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((num_objs,), dtype=torch.int64)  # All pedestrians are class 1
        masks = torch.as_tensor(masks, dtype=torch.uint8)
        
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        
        if self.transforms:
            img = self.transforms(img)
        else:
            img = transforms.ToTensor()(img)
        
        return img, target
    
    def __len__(self):
        return len(self.imgs)

print("Dataset class defined.")

In [ ]:
# Create dataset
dataset = PennFudanDataset('./data/PennFudanPed')
print(f"Total images: {len(dataset)}")

In [ ]:
# Visualize some samples
def visualize_sample(dataset, idx):
    img, target = dataset[idx]
    img_np = img.permute(1, 2, 0).numpy()
    
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(img_np)
    
    boxes = target['boxes']
    for box in boxes:
        xmin, ymin, xmax, ymax = box
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                             fill=False, edgecolor='red', linewidth=2)
        ax.add_patch(rect)
    
    ax.set_title(f'Image {idx}: {len(boxes)} pedestrians')
    plt.axis('off')
    plt.show()

# Show a few samples
for i in range(3):
    visualize_sample(dataset, i)

## 3. Split Data and Create DataLoaders

In [ ]:
def collate_fn(batch):
    """Custom collate function for object detection"""
    return tuple(zip(*batch))

# Split dataset
test_split = 0.2
test_size = int(len(dataset) * test_split)
train_size = len(dataset) - test_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 4. Create Faster R-CNN Model

We'll use a pre-trained Faster R-CNN model and fine-tune it for pedestrian detection.

In [ ]:
def get_model(num_classes):
    # Load pre-trained model
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

# Number of classes (background + pedestrian = 2)
num_classes = 2
model = get_model(num_classes)
model = model.to(device)

print(f"Model loaded with {num_classes} classes (background + pedestrian)")

## 5. Training Loop

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(data_loader, desc=f'Epoch {epoch}')
    for images, targets in progress_bar:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        progress_bar.set_postfix({'loss': f'{losses.item():.4f}'})
    
    return total_loss / len(data_loader)

print("Training function defined.")

In [ ]:
# Training parameters
num_epochs = 10
learning_rate = 0.005

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print(f"Starting training for {num_epochs} epochs...")

In [ ]:
# Train the model
train_losses = []

for epoch in range(1, num_epochs + 1):
    avg_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    train_losses.append(avg_loss)
    lr_scheduler.step()
    print(f'Epoch {epoch}: Average Loss = {avg_loss:.4f}')

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## 6. Evaluation and Visualization

In [ ]:
def visualize_predictions(model, dataset, idx, threshold=0.5):
    model.eval()
    img, target = dataset[idx]
    
    with torch.no_grad():
        prediction = model([img.to(device)])[0]
    
    img_np = img.permute(1, 2, 0).cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    
    # Ground truth
    axes[0].imshow(img_np)
    for box in target['boxes']:
        xmin, ymin, xmax, ymax = box.cpu().numpy()
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                             fill=False, edgecolor='green', linewidth=2)
        axes[0].add_patch(rect)
    axes[0].set_title(f'Ground Truth ({len(target["boxes"])} pedestrians)')
    axes[0].axis('off')
    
    # Predictions
    axes[1].imshow(img_np)
    scores = prediction['scores'].cpu().numpy()
    keep = scores > threshold
    boxes = prediction['boxes'].cpu().numpy()[keep]
    scores = scores[keep]
    
    for i, box in enumerate(boxes):
        xmin, ymin, xmax, ymax = box
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                             fill=False, edgecolor='red', linewidth=2)
        axes[1].add_patch(rect)
        axes[1].text(xmin, ymin - 5, f'{scores[i]:.2f}', color='red', fontsize=10)
    
    axes[1].set_title(f'Predictions ({len(boxes)} pedestrians, threshold={threshold})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions on test images
for i in range(5):
    visualize_predictions(model, test_dataset, i, threshold=0.5)

## 7. Calculate IoU and mAP

In [ ]:
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes [xmin, ymin, xmax, ymax]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def evaluate_model(model, dataset, iou_threshold=0.5, score_threshold=0.5):
    model.eval()
    all_precisions = []
    all_recalls = []
    
    for idx in range(len(dataset)):
        img, target = dataset[idx]
        
        with torch.no_grad():
            prediction = model([img.to(device)])[0]
        
        gt_boxes = target['boxes'].cpu().numpy()
        pred_boxes = prediction['boxes'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()
        
        # Filter by score threshold
        keep = scores > score_threshold
        pred_boxes = pred_boxes[keep]
        scores = scores[keep]
        
        if len(gt_boxes) == 0:
            continue
        
        # Calculate precision and recall for this image
        tp = 0
        matched = set()
        
        for pred_box in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in matched:
                    continue
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            if best_iou >= iou_threshold and best_gt_idx >= 0:
                tp += 1
                matched.add(best_gt_idx)
        
        precision = tp / len(pred_boxes) if len(pred_boxes) > 0 else 0
        recall = tp / len(gt_boxes) if len(gt_boxes) > 0 else 0
        all_precisions.append(precision)
        all_recalls.append(recall)
    
    return np.mean(all_precisions), np.mean(all_recalls)

# Evaluate
avg_precision, avg_recall = evaluate_model(model, test_dataset)
print(f'Average Precision: {avg_precision:.4f}')
print(f'Average Recall: {avg_recall:.4f}')

## 8. Save the Model

In [ ]:
# Save trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'num_classes': num_classes,
}, 'pedestrian_detection_model.pth')

print("Model saved to pedestrian_detection_model.pth")

In [ ]:
# Load model function
def load_model(path, num_classes=2):
    model = get_model(num_classes)
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    return model

print("Load function defined.")

## Summary

In this lab, we:
1. Downloaded the PennFudanPed pedestrian detection dataset
2. Created a custom PyTorch Dataset class with bounding box extraction from masks
3. Built a Faster R-CNN model with pre-trained weights
4. Fine-tuned the model for pedestrian detection
5. Evaluated the model with IoU, precision and recall metrics

### Next Steps
- Try different architectures (RetinaNet, SSD)
- Apply data augmentation
- Test on your own images